# Recycling VQA Colab Trainer (checkpoint / resume / model zoo / full-train x2)

이 노트북은 업로드하신 baseline을 **대회용으로 확장**한 버전입니다.

핵심 변경점
- **모델 선택을 설정값으로 분리**
- **QLoRA / LoRA / Full fine-tuning** 선택 가능
- **중간 체크포인트 저장 + resume**
- **검증 loss + 검증 accuracy** 추적
- **정답 생성(generate) / 4지선다 점수화(score choices)** 둘 다 지원
- **전체 train 재학습 2회 + 앙상블** 지원
- **객체 탐지 기반 crop 데이터셋 생성(옵션)** 코드 포함

권장 사용 순서
1. `MODEL_KEY="qwen2_5_vl_7b"`로 holdout 검증 1회
2. 설정 고정 후 `TRAIN_ON_FULL_DATA=True`
3. `FULL_TRAIN_ROUNDS=2`로 전체 데이터 2회 학습
4. 두 submission을 **majority vote 앙상블**


## 0. 설치

- `Qwen2.5-VL` / `Qwen3-VL` 지원을 위해 `transformers`는 소스 설치 기준으로 맞췄습니다.
- Colab 세션이 자주 끊기므로 **출력 경로는 Google Drive**로 잡는 것을 권장합니다.
- FlashAttention2는 **선택 옵션**입니다. 설치 실패가 잦으면 끄고 진행하세요.


In [ ]:
# ===== 0) Install =====
# Qwen3-VL / Qwen2.5-VL 모두 안정적으로 쓰기 위해 최신 transformers 소스 설치
!pip -q install -U git+https://github.com/huggingface/transformers accelerate
!pip -q install -U "peft>=0.13.2" "bitsandbytes>=0.46.1" datasets pillow pandas scikit-learn tqdm safetensors
!pip -q install -U torchvision torchaudio
# 옵션: flash-attn 이 필요한 경우 아래 주석 해제 (A100/H100 + 빌드 성공 시에만)
# !pip -q install -U flash-attn --no-build-isolation


In [ ]:
# ===== 1) Imports & Environment =====
import os
import re
import gc
import json
import math
import time
import copy
import glob
import shutil
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)

# Qwen family classes
from transformers import (
    Qwen2VLForConditionalGeneration,
    Qwen2_5_VLForConditionalGeneration,
)

# Qwen3-VL class는 최신 transformers 기준
try:
    from transformers import Qwen3VLForConditionalGeneration
    HAS_QWEN3 = True
except Exception:
    Qwen3VLForConditionalGeneration = None
    HAS_QWEN3 = False

# Colab drive (Colab이 아니면 무시)
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    drive = None
    IN_COLAB = False

Image.MAX_IMAGE_PIXELS = None
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BF16_AVAILABLE = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16_AVAILABLE else torch.float16

print("Device:", DEVICE)
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("cuDNN:", torch.backends.cudnn.version())
print("BF16 available:", BF16_AVAILABLE)
print("HAS_QWEN3:", HAS_QWEN3)


## 1. 설정

여기만 바꾸면 대부분의 실험을 제어할 수 있습니다.

### 추천 시작값
- **빠른 시작**: `qwen2_5_vl_3b`
- **기본 추천**: `qwen2_5_vl_7b`
- **고성능 후보**: `qwen3_vl_8b` (A100/H100 권장)
- **빠른 디버그**: `debug_max_samples=128`
- **최종 제출**: `TRAIN_ON_FULL_DATA=True`, `FULL_TRAIN_ROUNDS=2`


In [ ]:
# ===== 2) Config =====
MODEL_ZOO = {
    # 안정적/빠른 시작
    "qwen2_5_vl_3b": {
        "model_id": "Qwen/Qwen2.5-VL-3B-Instruct",
        "family": "qwen2_5_vl",
        "min_pixels": 256 * 28 * 28,
        "max_pixels": 768 * 28 * 28,
    },
    # baseline 대비 가장 추천하는 기본값
    "qwen2_5_vl_7b": {
        "model_id": "Qwen/Qwen2.5-VL-7B-Instruct",
        "family": "qwen2_5_vl",
        "min_pixels": 256 * 28 * 28,
        "max_pixels": 1024 * 28 * 28,
    },
    # 이전 버전 비교용
    "qwen2_vl_7b": {
        "model_id": "Qwen/Qwen2-VL-7B-Instruct",
        "family": "qwen2_vl",
        "min_pixels": 256 * 28 * 28,
        "max_pixels": 1024 * 28 * 28,
    },
    # 최신 계열 - high-end
    "qwen3_vl_2b": {
        "model_id": "Qwen/Qwen3-VL-2B-Instruct",
        "family": "qwen3_vl",
        "min_pixels": 256 * 28 * 28,
        "max_pixels": 768 * 28 * 28,
    },
    "qwen3_vl_4b": {
        "model_id": "Qwen/Qwen3-VL-4B-Instruct",
        "family": "qwen3_vl",
        "min_pixels": 256 * 28 * 28,
        "max_pixels": 1024 * 28 * 28,
    },
    "qwen3_vl_8b": {
        "model_id": "Qwen/Qwen3-VL-8B-Instruct",
        "family": "qwen3_vl",
        "min_pixels": 256 * 28 * 28,
        "max_pixels": 1280 * 28 * 28,
    },
}

@dataclass
class CFG:
    # ---------- run info ----------
    run_name: str = "recycling_vqa_qwen_holdout"
    seed: int = 42

    # ---------- data ----------
    data_zip_path: str = "/content/drive/MyDrive/SSAFY15/AI_challenge_260402/2026-ssafy-15-2-ai_dataset.zip"
    data_root: str = "/content"
    train_csv_name: str = "train.csv"
    test_csv_name: str = "test.csv"

    # ---------- outputs ----------
    output_root: str = "/content/drive/MyDrive/SSAFY15/AI_challenge_260402/runs"

    # ---------- model ----------
    model_key: str = "qwen2_5_vl_7b"
    train_strategy: str = "qlora"  # qlora / lora / full
    use_flash_attn: bool = False
    trust_remote_code: bool = True

    # ---------- LoRA ----------
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_bias: str = "none"
    lora_target_modules: Tuple[str, ...] = (
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    )

    # ---------- training ----------
    num_epochs: int = 2
    train_batch_size: int = 1
    valid_batch_size: int = 1
    grad_accum_steps: int = 8
    learning_rate: float = 2e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.03
    max_grad_norm: float = 1.0

    # ---------- split ----------
    val_ratio: float = 0.10
    stratify_by_answer: bool = True
    train_on_full_data: bool = False

    # ---------- eval / infer ----------
    eval_max_samples_for_acc: Optional[int] = None   # None이면 전체 valid
    infer_mode: str = "score"  # "generate" or "score"
    max_new_tokens: int = 2

    # ---------- checkpoint ----------
    save_every_n_optimizer_steps: int = 200
    keep_last_n_checkpoints: int = 3
    resume_from: Optional[str] = None  # None / "latest" / checkpoint_path

    # ---------- data augmentation ----------
    use_augmentation: bool = True
    random_hflip_p: float = 0.5
    color_jitter_p: float = 0.5
    affine_p: float = 0.3
    blur_p: float = 0.1

    # ---------- experiment controls ----------
    debug_max_samples: Optional[int] = None  # smoke test용
    num_workers: int = 2
    pin_memory: bool = True

    # ---------- final full-train ----------
    full_train_rounds: int = 2   # 이전 기수 힌트 반영
    do_submission_after_train: bool = True

    # ---------- object detection crop (optional) ----------
    use_object_crop_csv: bool = False
    object_crop_csv_path: Optional[str] = None

cfg = CFG()
cfg.model = MODEL_ZOO[cfg.model_key]
cfg.output_dir = os.path.join(cfg.output_root, cfg.run_name)

print(json.dumps({
    "run_name": cfg.run_name,
    "model_key": cfg.model_key,
    "model_id": cfg.model["model_id"],
    "train_strategy": cfg.train_strategy,
    "output_dir": cfg.output_dir,
    "infer_mode": cfg.infer_mode,
    "train_on_full_data": cfg.train_on_full_data,
}, indent=2, ensure_ascii=False))


## 2. 드라이브 마운트 / 데이터 압축 해제

- 체크포인트를 유지하려면 **Google Drive 마운트**를 먼저 하세요.
- 이미 압축 해제되어 있으면 unzip 셀은 건너뛰어도 됩니다.


In [ ]:
# ===== 3) Google Drive Mount =====
if IN_COLAB:
    drive.mount("/content/drive")
else:
    print("Not running in Colab; skip drive.mount().")


In [ ]:
# ===== 4) Unzip Dataset =====
zip_path = Path(cfg.data_zip_path)
if zip_path.exists():
    print(f"Found zip: {zip_path}")
    !unzip -qo "{zip_path}" -d "{cfg.data_root}"
else:
    print(f"[WARN] zip not found: {zip_path}")
    print("이미 /content 아래에 train.csv, test.csv, 이미지 폴더가 있으면 그대로 진행하세요.")


In [ ]:
# ===== 5) Utilities =====
SYSTEM_INSTRUCT = (
    "You are an expert visual question answering assistant for recycling and waste sorting. "
    "Answer using exactly one lowercase letter among a, b, c, or d. No explanation."
)

CHOICES = ["a", "b", "c", "d"]

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def ensure_dir(path: str) -> str:
    os.makedirs(path, exist_ok=True)
    return path

def normalize_answer(x: Any) -> str:
    x = str(x).strip().lower()
    return x if x in CHOICES else "a"

def extract_choice(text: str) -> str:
    text = str(text).strip().lower()
    if text in CHOICES:
        return text
    matches = re.findall(r"\b([abcd])\b", text)
    if matches:
        return matches[-1]
    return "a"

def build_mc_prompt(question: Any, a: Any, b: Any, c: Any, d: Any) -> str:
    return (
        f"{str(question).strip()}\n"
        f"(a) {str(a).strip()}\n"
        f"(b) {str(b).strip()}\n"
        f"(c) {str(c).strip()}\n"
        f"(d) {str(d).strip()}\n\n"
        "반드시 a, b, c, d 중 하나의 소문자 한 글자만 출력하세요."
    )

def exif_rgb(image: Image.Image) -> Image.Image:
    image = ImageOps.exif_transpose(image)
    return image.convert("RGB")

def load_image(path: str) -> Image.Image:
    img = Image.open(path)
    return exif_rgb(img)

def resolve_path(path_str: str, data_root: str) -> str:
    p = Path(str(path_str))
    if p.is_absolute() and p.exists():
        return str(p)
    cands = [
        Path(data_root) / p,
        Path("/content") / p,
        Path.cwd() / p,
    ]
    for cand in cands:
        if cand.exists():
            return str(cand)
    return str(p)

def autocast_device_type(device: torch.device) -> str:
    return "cuda" if device.type == "cuda" else "cpu"

def cleanup_old_checkpoints(
    ckpt_root: str,
    keep_last_n: int,
    protected: Optional[List[str]] = None,
) -> None:
    protected = set(protected or [])
    if not os.path.exists(ckpt_root):
        return
    ckpts = []
    for d in os.listdir(ckpt_root):
        full = os.path.join(ckpt_root, d)
        if os.path.isdir(full) and d.startswith("step_"):
            m = re.search(r"step_(\d+)", d)
            step = int(m.group(1)) if m else -1
            ckpts.append((step, full))
    ckpts.sort(key=lambda x: x[0])
    removable = [p for _, p in ckpts[:-keep_last_n]]
    for p in removable:
        if p in protected:
            continue
        shutil.rmtree(p, ignore_errors=True)

def get_latest_checkpoint(run_dir: str) -> Optional[str]:
    ckpt_root = os.path.join(run_dir, "checkpoints")
    if not os.path.exists(ckpt_root):
        return None
    step_dirs = []
    for d in os.listdir(ckpt_root):
        full = os.path.join(ckpt_root, d)
        if os.path.isdir(full) and d.startswith("step_"):
            m = re.search(r"step_(\d+)", d)
            if m:
                step_dirs.append((int(m.group(1)), full))
    if not step_dirs:
        return None
    step_dirs.sort(key=lambda x: x[0])
    return step_dirs[-1][1]

def save_json(data: Dict[str, Any], path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def save_checkpoint(
    run_dir: str,
    model,
    processor,
    optimizer,
    scheduler,
    scaler,
    epoch: int,
    step_in_epoch: int,
    global_step: int,
    best_val_acc: float,
    best_ckpt_path: Optional[str],
    cfg_obj: CFG,
) -> str:
    ckpt_root = ensure_dir(os.path.join(run_dir, "checkpoints"))
    ckpt_dir = ensure_dir(os.path.join(ckpt_root, f"step_{global_step:07d}"))

    model.save_pretrained(ckpt_dir)
    processor.save_pretrained(ckpt_dir)

    trainer_state = {
        "epoch": epoch,
        "step_in_epoch": step_in_epoch,
        "global_step": global_step,
        "best_val_acc": best_val_acc,
        "best_ckpt_path": best_ckpt_path,
        "optimizer": optimizer.state_dict() if optimizer is not None else None,
        "scheduler": scheduler.state_dict() if scheduler is not None else None,
        "scaler": scaler.state_dict() if scaler is not None else None,
        "cfg": asdict(cfg_obj),
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    torch.save(trainer_state, os.path.join(ckpt_dir, "trainer_state.pt"))

    protected = [best_ckpt_path] if best_ckpt_path else []
    cleanup_old_checkpoints(
        ckpt_root,
        keep_last_n=cfg_obj.keep_last_n_checkpoints,
        protected=protected,
    )
    return ckpt_dir

def maybe_load_resume_checkpoint(run_dir: str, resume_from: Optional[str]) -> Optional[str]:
    if resume_from is None:
        return None
    if resume_from == "latest":
        return get_latest_checkpoint(run_dir)
    if os.path.exists(resume_from):
        return resume_from
    print(f"[WARN] resume checkpoint not found: {resume_from}")
    return None

def count_trainable_params(model) -> Tuple[int, int]:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


In [ ]:
# ===== 6) Data Load / Clean =====
set_seed(cfg.seed)
ensure_dir(cfg.output_dir)

train_csv_path = Path(cfg.data_root) / cfg.train_csv_name
test_csv_path  = Path(cfg.data_root) / cfg.test_csv_name

if not train_csv_path.exists():
    raise FileNotFoundError(f"train csv not found: {train_csv_path}")
if not test_csv_path.exists():
    raise FileNotFoundError(f"test csv not found: {test_csv_path}")

train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

required_train_cols = ["path", "question", "a", "b", "c", "d", "answer"]
required_test_cols = ["id", "path", "question", "a", "b", "c", "d"]

for col in required_train_cols:
    if col not in train_df.columns:
        raise ValueError(f"train.csv missing column: {col}")
for col in required_test_cols:
    if col not in test_df.columns:
        raise ValueError(f"test.csv missing column: {col}")

train_df = train_df.copy()
test_df = test_df.copy()

train_df["path"] = train_df["path"].map(lambda x: resolve_path(x, cfg.data_root))
test_df["path"] = test_df["path"].map(lambda x: resolve_path(x, cfg.data_root))
train_df["answer"] = train_df["answer"].map(normalize_answer)

# 중복 제거
dedup_cols = ["path", "question", "a", "b", "c", "d", "answer"]
before_n = len(train_df)
train_df = train_df.drop_duplicates(subset=dedup_cols).reset_index(drop=True)
after_n = len(train_df)

# 디버그 샘플
if cfg.debug_max_samples is not None:
    train_df = train_df.sample(
        n=min(cfg.debug_max_samples, len(train_df)),
        random_state=cfg.seed
    ).reset_index(drop=True)

# 옵션: 객체 탐지 crop csv 합치기
if cfg.use_object_crop_csv and cfg.object_crop_csv_path is not None and os.path.exists(cfg.object_crop_csv_path):
    crop_df = pd.read_csv(cfg.object_crop_csv_path)
    if "path" not in crop_df.columns:
        raise ValueError("object crop csv must contain 'path' column")
    crop_df["path"] = crop_df["path"].map(lambda x: resolve_path(x, cfg.data_root))
    if "answer" in crop_df.columns:
        crop_df["answer"] = crop_df["answer"].map(normalize_answer)
    # train 필드만 남기고 원본 train 컬럼 순서 맞춤
    crop_df = crop_df[[c for c in train_df.columns if c in crop_df.columns]]
    train_df = pd.concat([train_df, crop_df], ignore_index=True)

# 경로 존재 여부 확인
missing_train = (~train_df["path"].map(lambda x: Path(x).exists())).sum()
missing_test = (~test_df["path"].map(lambda x: Path(x).exists())).sum()

print(f"train size: {len(train_df):,} (dedup removed {before_n - after_n:,})")
print(f"test size : {len(test_df):,}")
print(f"missing train images: {missing_train}")
print(f"missing test images : {missing_test}")
display(train_df.head(2))


## 3. Holdout split / Full-train 분기

- **모델 선정 단계**: `train_on_full_data=False`
- **최종 제출 단계**: `train_on_full_data=True`
- 검증 정확도는 `score choices` 방식으로 보는 것을 권장합니다.


In [ ]:
# ===== 7) Split =====
if cfg.train_on_full_data:
    train_split_df = train_df.reset_index(drop=True)
    valid_split_df = None
    print("[Mode] FULL TRAIN on all training rows")
else:
    stratify_col = train_df["answer"] if cfg.stratify_by_answer else None
    train_split_df, valid_split_df = train_test_split(
        train_df,
        test_size=cfg.val_ratio,
        random_state=cfg.seed,
        stratify=stratify_col,
    )
    train_split_df = train_split_df.reset_index(drop=True)
    valid_split_df = valid_split_df.reset_index(drop=True)
    print("[Mode] HOLDOUT VALIDATION")
    print("train:", len(train_split_df), "valid:", len(valid_split_df))
    print(valid_split_df["answer"].value_counts(normalize=True).sort_index())


In [ ]:
# ===== 8) Augmentation =====
def build_train_transform(cfg: CFG):
    transforms_list = []
    if cfg.use_augmentation:
        transforms_list.append(T.RandomHorizontalFlip(p=cfg.random_hflip_p))
        transforms_list.append(T.RandomApply(
            [T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02)],
            p=cfg.color_jitter_p
        ))
        transforms_list.append(T.RandomApply(
            [T.RandomAffine(degrees=5, translate=(0.03, 0.03), scale=(0.95, 1.05), shear=2)],
            p=cfg.affine_p
        ))
        transforms_list.append(T.RandomApply(
            [T.GaussianBlur(kernel_size=3)],
            p=cfg.blur_p
        ))
    return T.Compose(transforms_list) if transforms_list else None

train_transform = build_train_transform(cfg)
valid_transform = None

print(train_transform)


In [ ]:
# ===== 9) Dataset / Collator =====
def build_prompt_messages(row: pd.Series, image: Image.Image) -> List[Dict[str, Any]]:
    user_text = build_mc_prompt(
        row["question"], row["a"], row["b"], row["c"], row["d"]
    )
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": user_text},
            ],
        },
    ]
    return messages

def build_train_messages(row: pd.Series, image: Image.Image) -> List[Dict[str, Any]]:
    messages = build_prompt_messages(row, image)
    messages.append({
        "role": "assistant",
        "content": [{"type": "text", "text": normalize_answer(row["answer"])}],
    })
    return messages

class RecyclingVQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, train: bool = True, transform=None):
        self.df = df.reset_index(drop=True)
        self.train = train
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        image = load_image(row["path"])
        if self.transform is not None:
            image = self.transform(image)

        sample = {
            "row": row.to_dict(),
            "image": image,
            "prompt_messages": build_prompt_messages(row, image),
        }
        if self.train:
            sample["answer"] = normalize_answer(row["answer"])
            sample["full_messages"] = build_train_messages(row, image)
        return sample

@dataclass
class RecyclingCollator:
    processor: Any
    train: bool = True
    assistant_only_loss: bool = True

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        texts, images = [], []
        rows, answers = [], []

        for sample in batch:
            rows.append(sample["row"])
            answers.append(sample.get("answer"))
            images.append(sample["image"])

            if self.train:
                text = self.processor.apply_chat_template(
                    sample["full_messages"],
                    tokenize=False,
                    add_generation_prompt=False,
                )
            else:
                text = self.processor.apply_chat_template(
                    sample["prompt_messages"],
                    tokenize=False,
                    add_generation_prompt=True,
                )
            texts.append(text)

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        enc["rows"] = rows
        enc["answers"] = answers

        if self.train:
            labels = enc["input_ids"].clone()
            labels[enc["attention_mask"] == 0] = -100

            if self.assistant_only_loss:
                # prompt 구간을 -100 처리하여 "정답 토큰"만 학습
                for i, sample in enumerate(batch):
                    prompt_text = self.processor.apply_chat_template(
                        sample["prompt_messages"],
                        tokenize=False,
                        add_generation_prompt=True,
                    )
                    prompt_enc = self.processor(
                        text=[prompt_text],
                        images=[sample["image"]],
                        return_tensors="pt",
                    )
                    prompt_len = prompt_enc["input_ids"].shape[1]
                    labels[i, :prompt_len] = -100

            enc["labels"] = labels

        return enc

def move_batch_to_device(batch: Dict[str, Any], device: torch.device):
    tensor_batch = {}
    meta_batch = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            tensor_batch[k] = v.to(device)
        else:
            meta_batch[k] = v
    return tensor_batch, meta_batch


In [ ]:
# ===== 10) Model / Processor Loader =====
def get_model_class(model_family: str):
    if model_family == "qwen2_vl":
        return Qwen2VLForConditionalGeneration
    if model_family == "qwen2_5_vl":
        return Qwen2_5_VLForConditionalGeneration
    if model_family == "qwen3_vl":
        if not HAS_QWEN3:
            raise ImportError(
                "Qwen3VLForConditionalGeneration is not available. "
                "Please reinstall latest transformers from source."
            )
        return Qwen3VLForConditionalGeneration
    raise ValueError(f"Unsupported model family: {model_family}")

def load_model_and_processor(cfg: CFG, adapter_or_ckpt_dir: Optional[str] = None):
    model_id = cfg.model["model_id"]
    model_family = cfg.model["family"]
    model_cls = get_model_class(model_family)

    processor_source = adapter_or_ckpt_dir if (adapter_or_ckpt_dir is not None and os.path.exists(adapter_or_ckpt_dir)) else model_id
    processor = AutoProcessor.from_pretrained(
        processor_source,
        min_pixels=cfg.model["min_pixels"],
        max_pixels=cfg.model["max_pixels"],
        trust_remote_code=cfg.trust_remote_code,
    )

    # 학습 시 오른쪽 패딩 고정
    if hasattr(processor, "tokenizer"):
        processor.tokenizer.padding_side = "right"

    use_4bit = cfg.train_strategy == "qlora"
    quantization_config = None
    if use_4bit:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=AMP_DTYPE,
        )

    model_source = model_id
    # full fine-tuning이면 체크포인트/최종 디렉토리에서 직접 다시 로드
    if cfg.train_strategy == "full" and adapter_or_ckpt_dir is not None and os.path.exists(adapter_or_ckpt_dir):
        model_source = adapter_or_ckpt_dir

    model_kwargs = {
        "device_map": "auto",
        "torch_dtype": AMP_DTYPE,
        "trust_remote_code": cfg.trust_remote_code,
    }
    if cfg.use_flash_attn:
        model_kwargs["attn_implementation"] = "flash_attention_2"
    else:
        model_kwargs["attn_implementation"] = "sdpa"
    if quantization_config is not None:
        model_kwargs["quantization_config"] = quantization_config

    # base model load
    base_model = model_cls.from_pretrained(model_source, **model_kwargs)
    base_model.config.use_cache = False
    base_model.gradient_checkpointing_enable()

    if cfg.train_strategy in ["qlora", "lora"]:
        if cfg.train_strategy == "qlora":
            base_model = prepare_model_for_kbit_training(base_model)

        lora_cfg = LoraConfig(
            r=cfg.lora_r,
            lora_alpha=cfg.lora_alpha,
            lora_dropout=cfg.lora_dropout,
            bias=cfg.lora_bias,
            task_type="CAUSAL_LM",
            target_modules=list(cfg.lora_target_modules),
        )

        if adapter_or_ckpt_dir is not None and os.path.exists(adapter_or_ckpt_dir):
            model = PeftModel.from_pretrained(
                base_model,
                adapter_or_ckpt_dir,
                is_trainable=True,
            )
        else:
            model = get_peft_model(base_model, lora_cfg)
    elif cfg.train_strategy == "full":
        model = base_model
        for p in model.parameters():
            p.requires_grad = True
    else:
        raise ValueError(f"Unsupported train_strategy: {cfg.train_strategy}")

    trainable, total = count_trainable_params(model)
    print(f"Model: {model_source}")
    print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.4f}%)")
    return model, processor

# smoke test (실제 학습 전 모델만 바꾸고 싶을 때 확인용)
print(cfg.model["model_id"], cfg.model["family"])


In [ ]:
# ===== 11) Dataloader Builders =====
def make_train_loader(df: pd.DataFrame, processor, cfg: CFG, epoch_seed: int):
    ds = RecyclingVQADataset(
        df,
        train=True,
        transform=build_train_transform(cfg),
    )
    gen = torch.Generator()
    gen.manual_seed(epoch_seed)
    loader = DataLoader(
        ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        generator=gen,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        collate_fn=RecyclingCollator(processor=processor, train=True, assistant_only_loss=True),
    )
    return loader

def make_valid_loader(df: Optional[pd.DataFrame], processor, cfg: CFG):
    if df is None:
        return None
    ds = RecyclingVQADataset(
        df,
        train=True,
        transform=None,
    )
    loader = DataLoader(
        ds,
        batch_size=cfg.valid_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        collate_fn=RecyclingCollator(processor=processor, train=True, assistant_only_loss=True),
    )
    return loader


In [ ]:
# ===== 12) MCQ Inference Helpers =====
@torch.no_grad()
def predict_choice_generate(model, processor, row: pd.Series, device: torch.device, max_new_tokens: int = 2) -> str:
    image = load_image(row["path"])
    messages = build_prompt_messages(row, image)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()
    with torch.autocast(
        device_type=autocast_device_type(device),
        dtype=AMP_DTYPE,
        enabled=(device.type == "cuda"),
    ):
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=processor.tokenizer.eos_token_id,
        )
    decoded = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    return extract_choice(decoded)

@torch.no_grad()
def score_single_choice_from_image(
    model,
    processor,
    row: pd.Series,
    image: Image.Image,
    answer_choice: str,
    device: torch.device,
) -> float:
    # answer_choice 하나에 대한 conditional loss 계산
    prompt_messages = build_prompt_messages(row, image)
    full_messages = prompt_messages + [{
        "role": "assistant",
        "content": [{"type": "text", "text": answer_choice}],
    }]

    prompt_text = processor.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = processor.apply_chat_template(
        full_messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    enc = processor(text=[full_text], images=[image], return_tensors="pt")
    prompt_enc = processor(text=[prompt_text], images=[image], return_tensors="pt")

    labels = enc["input_ids"].clone()
    labels[:, :prompt_enc["input_ids"].shape[1]] = -100
    labels[enc["attention_mask"] == 0] = -100

    enc = {k: v.to(device) for k, v in enc.items()}
    labels = labels.to(device)

    model.eval()
    with torch.autocast(
        device_type=autocast_device_type(device),
        dtype=AMP_DTYPE,
        enabled=(device.type == "cuda"),
    ):
        outputs = model(**enc, labels=labels)
    return float(outputs.loss.item())

@torch.no_grad()
def predict_choice_score(model, processor, row: pd.Series, device: torch.device) -> str:
    image = load_image(row["path"])
    scores = {
        c: score_single_choice_from_image(model, processor, row, image, c, device)
        for c in CHOICES
    }
    # loss가 가장 낮은 선택지
    pred = min(scores.items(), key=lambda x: x[1])[0]
    return pred

def predict_choice(model, processor, row: pd.Series, device: torch.device, infer_mode: str, max_new_tokens: int = 2) -> str:
    if infer_mode == "generate":
        return predict_choice_generate(model, processor, row, device, max_new_tokens=max_new_tokens)
    if infer_mode == "score":
        return predict_choice_score(model, processor, row, device)
    raise ValueError(f"Unsupported infer_mode: {infer_mode}")

@torch.no_grad()
def evaluate_mcq_accuracy(
    model,
    processor,
    valid_df: Optional[pd.DataFrame],
    device: torch.device,
    infer_mode: str = "score",
    max_samples: Optional[int] = None,
    max_new_tokens: int = 2,
) -> Optional[float]:
    if valid_df is None or len(valid_df) == 0:
        return None

    eval_df = valid_df
    if max_samples is not None and len(eval_df) > max_samples:
        eval_df = eval_df.sample(n=max_samples, random_state=cfg.seed).reset_index(drop=True)

    correct = 0
    total = 0
    for _, row in eval_df.iterrows():
        pred = predict_choice(
            model,
            processor,
            row,
            device,
            infer_mode=infer_mode,
            max_new_tokens=max_new_tokens,
        )
        gold = normalize_answer(row["answer"])
        correct += int(pred == gold)
        total += 1

    return correct / max(total, 1)


In [ ]:
# ===== 13) Training / Validation =====
def build_optimizer(model, cfg: CFG):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )
    return optimizer

def run_validation_loss(model, valid_loader, device: torch.device) -> Optional[float]:
    if valid_loader is None:
        return None

    model.eval()
    losses = []
    for batch in valid_loader:
        tensor_batch, _ = move_batch_to_device(batch, device)
        with torch.no_grad():
            with torch.autocast(device_type=autocast_device_type(device), dtype=AMP_DTYPE, enabled=(device.type == "cuda")):
                outputs = model(**tensor_batch)
        losses.append(outputs.loss.item())

    if not losses:
        return None
    return float(np.mean(losses))

def train_one_run(
    cfg: CFG,
    train_df: pd.DataFrame,
    valid_df: Optional[pd.DataFrame],
    run_dir: str,
):
    set_seed(cfg.seed)
    ensure_dir(run_dir)

    resume_ckpt = maybe_load_resume_checkpoint(run_dir, cfg.resume_from)
    model, processor = load_model_and_processor(cfg, adapter_or_ckpt_dir=resume_ckpt)

    valid_loader = make_valid_loader(valid_df, processor, cfg)
    # train_loader는 epoch별로 섞기 위해 매 epoch 생성

    # update step 수 계산
    temp_train_loader = make_train_loader(train_df, processor, cfg, epoch_seed=cfg.seed)
    num_update_steps_per_epoch = math.ceil(len(temp_train_loader) / cfg.grad_accum_steps)
    del temp_train_loader
    gc.collect()

    total_training_steps = cfg.num_epochs * num_update_steps_per_epoch
    optimizer = build_optimizer(model, cfg)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, int(total_training_steps * cfg.warmup_ratio)),
        num_training_steps=max(1, total_training_steps),
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda" and AMP_DTYPE == torch.float16))

    start_epoch = 0
    resume_step_in_epoch = 0
    global_step = 0
    best_val_acc = -1.0
    best_ckpt_path = None

    # optimizer / scheduler / scaler state 복원
    if resume_ckpt is not None:
        trainer_state_path = os.path.join(resume_ckpt, "trainer_state.pt")
        if os.path.exists(trainer_state_path):
            state = torch.load(trainer_state_path, map_location="cpu")
            if state.get("optimizer") is not None:
                optimizer.load_state_dict(state["optimizer"])
            if state.get("scheduler") is not None:
                scheduler.load_state_dict(state["scheduler"])
            if state.get("scaler") is not None and scaler.is_enabled():
                scaler.load_state_dict(state["scaler"])
            start_epoch = int(state.get("epoch", 0))
            resume_step_in_epoch = int(state.get("step_in_epoch", 0))
            global_step = int(state.get("global_step", 0))
            best_val_acc = float(state.get("best_val_acc", -1.0))
            best_ckpt_path = state.get("best_ckpt_path", None)

            # epoch 마지막 step까지 저장된 상태면 다음 epoch부터 시작
            if resume_step_in_epoch >= len(make_train_loader(train_df, processor, cfg, epoch_seed=cfg.seed + start_epoch)):
                start_epoch += 1
                resume_step_in_epoch = 0

            print(f"[Resume] from {resume_ckpt}")
            print(f"start_epoch={start_epoch}, resume_step_in_epoch={resume_step_in_epoch}, global_step={global_step}, best_val_acc={best_val_acc:.4f}")

    metrics_log_path = os.path.join(run_dir, "metrics.jsonl")
    model.train()

    for epoch in range(start_epoch, cfg.num_epochs):
        epoch_seed = cfg.seed + epoch
        train_loader = make_train_loader(train_df, processor, cfg, epoch_seed=epoch_seed)

        running_loss = 0.0
        seen_update = 0
        optimizer.zero_grad(set_to_none=True)

        print(f"\n===== Epoch {epoch + 1} / {cfg.num_epochs} =====")
        for step, batch in enumerate(train_loader, start=1):
            if epoch == start_epoch and step <= resume_step_in_epoch:
                continue

            tensor_batch, _ = move_batch_to_device(batch, DEVICE)

            with torch.autocast(device_type=autocast_device_type(DEVICE), dtype=AMP_DTYPE, enabled=(DEVICE.type == "cuda")):
                outputs = model(**tensor_batch)
                loss = outputs.loss / cfg.grad_accum_steps

            if scaler.is_enabled():
                scaler.scale(loss).backward()
            else:
                loss.backward()

            running_loss += float(loss.item())

            should_update = (step % cfg.grad_accum_steps == 0) or (step == len(train_loader))
            if should_update:
                if scaler.is_enabled():
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                    optimizer.step()

                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

                global_step += 1
                seen_update += 1
                avg_loss = running_loss / max(1, seen_update)

                if global_step % 10 == 0:
                    print(f"[train] epoch={epoch+1} step={step}/{len(train_loader)} opt_step={global_step} avg_loss={avg_loss:.4f}")

                if global_step % cfg.save_every_n_optimizer_steps == 0:
                    ckpt_path = save_checkpoint(
                        run_dir=run_dir,
                        model=model,
                        processor=processor,
                        optimizer=optimizer,
                        scheduler=scheduler,
                        scaler=scaler,
                        epoch=epoch,
                        step_in_epoch=step,
                        global_step=global_step,
                        best_val_acc=best_val_acc,
                        best_ckpt_path=best_ckpt_path,
                        cfg_obj=cfg,
                    )
                    print(f"[ckpt] saved: {ckpt_path}")

        # epoch end validation
        val_loss = run_validation_loss(model, valid_loader, DEVICE)
        val_acc = evaluate_mcq_accuracy(
            model,
            processor,
            valid_df,
            DEVICE,
            infer_mode=cfg.infer_mode,
            max_samples=cfg.eval_max_samples_for_acc,
            max_new_tokens=cfg.max_new_tokens,
        )

        epoch_summary = {
            "epoch": epoch + 1,
            "global_step": global_step,
            "train_avg_loss": running_loss / max(1, seen_update),
            "val_loss": val_loss,
            "val_acc": val_acc,
            "time": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        with open(metrics_log_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(epoch_summary, ensure_ascii=False) + "\n")

        print(f"[epoch end] {json.dumps(epoch_summary, ensure_ascii=False)}")

        # best checkpoint 갱신
        improved = (val_acc is not None) and (val_acc > best_val_acc)
        if improved:
            best_val_acc = val_acc
            best_ckpt_path = save_checkpoint(
                run_dir=run_dir,
                model=model,
                processor=processor,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                epoch=epoch,
                step_in_epoch=len(train_loader),
                global_step=global_step,
                best_val_acc=best_val_acc,
                best_ckpt_path=best_ckpt_path,
                cfg_obj=cfg,
            )
            print(f"[best] val_acc improved -> {best_val_acc:.4f}, checkpoint: {best_ckpt_path}")

        # 매 epoch 마지막 상태 저장
        last_ckpt_path = save_checkpoint(
            run_dir=run_dir,
            model=model,
            processor=processor,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            epoch=epoch,
            step_in_epoch=len(train_loader),
            global_step=global_step,
            best_val_acc=best_val_acc,
            best_ckpt_path=best_ckpt_path,
            cfg_obj=cfg,
        )
        print(f"[last] checkpoint saved: {last_ckpt_path}")

    final_dir = ensure_dir(os.path.join(run_dir, "final"))
    model.save_pretrained(final_dir)
    processor.save_pretrained(final_dir)
    print(f"[final] saved to: {final_dir}")

    summary = {
        "run_dir": run_dir,
        "final_dir": final_dir,
        "best_ckpt_path": best_ckpt_path,
        "best_val_acc": best_val_acc if best_val_acc >= 0 else None,
        "metrics_log_path": metrics_log_path,
    }
    save_json(summary, os.path.join(run_dir, "run_summary.json"))
    return summary


In [ ]:
# ===== 14) Inference / Submission =====
def run_inference_to_submission(
    cfg: CFG,
    model_dir: str,
    test_df: pd.DataFrame,
    submission_name: str,
) -> str:
    # 추론은 학습 모드와 동일 전략으로 모델 로드
    infer_cfg = copy.deepcopy(cfg)
    # adapter/final dir 로딩
    model, processor = load_model_and_processor(infer_cfg, adapter_or_ckpt_dir=model_dir)
    model.eval()

    preds = []
    for _, row in test_df.iterrows():
        pred = predict_choice(model, processor, row, DEVICE, infer_mode=cfg.infer_mode, max_new_tokens=cfg.max_new_tokens)
        preds.append(pred)

    submission = pd.DataFrame({
        "id": test_df["id"],
        "answer": preds,
    })

    submission_dir = ensure_dir(os.path.join(cfg.output_dir, "submissions"))
    out_path = os.path.join(submission_dir, submission_name)
    submission.to_csv(out_path, index=False)
    print(f"[submission] saved: {out_path}")
    return out_path

def ensemble_submission_files(csv_paths: List[str], out_path: str) -> str:
    if len(csv_paths) == 0:
        raise ValueError("csv_paths is empty")

    dfs = [pd.read_csv(p) for p in csv_paths]
    base = dfs[0][["id"]].copy()

    vote_df = pd.DataFrame({f"pred_{i}": df["answer"].astype(str) for i, df in enumerate(dfs)})

    def majority_vote(row):
        counts = row.value_counts()
        # tie면 앞 submission 우선
        max_cnt = counts.max()
        top = set(counts[counts == max_cnt].index.tolist())
        for v in row.tolist():
            if v in top:
                return v
        return row.iloc[0]

    base["answer"] = vote_df.apply(majority_vote, axis=1)
    base.to_csv(out_path, index=False)
    print(f"[ensemble] saved: {out_path}")
    return out_path


## 4. 실행 셀

### 일반적인 사용법
- **모델 선택 실험**: `train_on_full_data=False`
- **최종 제출용**: `train_on_full_data=True`, `full_train_rounds=2`
- 첫 실험은 `qwen2_5_vl_7b + qlora + infer_mode="score"` 추천


In [ ]:
# ===== 15) Run Single Holdout Experiment =====
# 모델/파라미터 탐색 단계에서 먼저 실행
summary = train_one_run(
    cfg=cfg,
    train_df=train_split_df,
    valid_df=valid_split_df,
    run_dir=cfg.output_dir,
)
summary


In [ ]:
# ===== 16) Optional: Submission After Current Run =====
# holdout 단계에서는 보통 best checkpoint 또는 final 중 하나로 제출 파일만 확인
if cfg.do_submission_after_train:
    infer_model_dir = summary["best_ckpt_path"] or summary["final_dir"]
    submission_path = run_inference_to_submission(
        cfg=cfg,
        model_dir=infer_model_dir,
        test_df=test_df,
        submission_name=f"{cfg.run_name}_{cfg.model_key}_{cfg.infer_mode}.csv",
    )
    print(submission_path)


## 5. 전체 데이터 2회 full-train + 앙상블

이전 기수 힌트를 반영해 넣은 셀입니다.

권장 흐름
1. holdout에서 모델/하이퍼파라미터 확정
2. 아래 셀에서 `train_on_full_data=True`
3. seed만 바꿔 **2회 재학습**
4. 두 submission majority vote


In [ ]:
# ===== 17) Optional: Full-Train x2 =====
# holdout에서 설정 확정 후 사용
DO_FULL_TRAIN_ROUNDS = False  # True 로 바꿔서 실행

if DO_FULL_TRAIN_ROUNDS:
    full_cfg = copy.deepcopy(cfg)
    full_cfg.train_on_full_data = True
    full_cfg.resume_from = None

    full_train_df = train_df.reset_index(drop=True)
    submission_paths = []

    for round_idx in range(full_cfg.full_train_rounds):
        round_cfg = copy.deepcopy(full_cfg)
        round_cfg.seed = full_cfg.seed + round_idx
        round_cfg.run_name = f"{cfg.run_name}_full_round{round_idx+1}"
        round_cfg.output_dir = os.path.join(round_cfg.output_root, round_cfg.run_name)

        print(f"\n######## FULL TRAIN ROUND {round_idx + 1} / {full_cfg.full_train_rounds} ########")
        round_summary = train_one_run(
            cfg=round_cfg,
            train_df=full_train_df,
            valid_df=None,
            run_dir=round_cfg.output_dir,
        )

        round_submission = run_inference_to_submission(
            cfg=round_cfg,
            model_dir=round_summary["final_dir"],
            test_df=test_df,
            submission_name=f"{round_cfg.run_name}_{round_cfg.model_key}_{round_cfg.infer_mode}.csv",
        )
        submission_paths.append(round_submission)

    ensemble_dir = ensure_dir(os.path.join(cfg.output_root, "ensembles"))
    ensemble_path = os.path.join(ensemble_dir, f"{cfg.run_name}_fulltrain_vote.csv")
    ensemble_submission_files(submission_paths, ensemble_path)
    print("submission_paths:", submission_paths)
    print("ensemble_path:", ensemble_path)
else:
    print("Set DO_FULL_TRAIN_ROUNDS=True to run full-train x2 pipeline.")


## 6. 선택 옵션: 객체 탐지로 crop 데이터 생성

아이디어
- 원본 이미지 전체 장면 + 객체 중심 crop 둘 다 학습시키면,
  작은 폐기물/라벨/캡 같은 **세부 시각 단서**를 더 잘 보게 만들 수 있습니다.
- 아래 셀은 **Grounding DINO**를 사용해 crop 이미지를 저장하고, 나중에 train에 합칠 수 있는 CSV를 만듭니다.

권장 사용법
1. 텍스트 쿼리를 재활용 도메인에 맞게 수정
2. 일부 샘플에서 crop 품질 수동 확인
3. `cfg.use_object_crop_csv=True`로 학습에 합치기


In [ ]:
# ===== 18) Optional: Build Object-Crop CSV with Grounding DINO =====
# 느릴 수 있으므로 필요한 경우에만 실행
BUILD_OBJECT_CROP = False

if BUILD_OBJECT_CROP:
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

    det_model_id = "IDEA-Research/grounding-dino-tiny"
    det_processor = AutoProcessor.from_pretrained(det_model_id)
    det_model = AutoModelForZeroShotObjectDetection.from_pretrained(det_model_id).to(DEVICE)

    # 재활용/분리배출 도메인에 맞는 query 예시
    TEXT_QUERIES = [[
        "plastic bottle",
        "can",
        "glass bottle",
        "paper cup",
        "carton",
        "plastic bag",
        "vinyl bag",
        "styrofoam",
        "battery",
        "label",
        "cap",
        "straw",
        "food waste",
        "trash",
    ]]

    crop_root = ensure_dir(os.path.join(cfg.output_dir, "detected_crops"))
    crop_rows = []

    def expand_box(box, width, height, expand_ratio=0.08):
        x1, y1, x2, y2 = box
        bw = x2 - x1
        bh = y2 - y1
        x1 = max(0, x1 - bw * expand_ratio)
        y1 = max(0, y1 - bh * expand_ratio)
        x2 = min(width, x2 + bw * expand_ratio)
        y2 = min(height, y2 + bh * expand_ratio)
        return int(x1), int(y1), int(x2), int(y2)

    source_df = train_df.copy()
    for idx, row in source_df.iterrows():
        image = load_image(row["path"])
        inputs = det_processor(images=image, text=TEXT_QUERIES, return_tensors="pt").to(DEVICE)

        with torch.no_grad():
            outputs = det_model(**inputs)

        results = det_processor.post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            threshold=0.30,
            text_threshold=0.25,
            target_sizes=[(image.height, image.width)],
        )[0]

        if len(results["boxes"]) == 0:
            continue

        # 가장 높은 score 1개만 사용
        best_idx = int(torch.argmax(results["scores"]).item())
        box = results["boxes"][best_idx].tolist()
        score = float(results["scores"][best_idx].item())
        label = results["labels"][best_idx] if "labels" in results else "object"

        x1, y1, x2, y2 = expand_box(box, image.width, image.height, expand_ratio=0.08)
        crop = image.crop((x1, y1, x2, y2))

        crop_name = f"crop_{idx:06d}.jpg"
        crop_path = os.path.join(crop_root, crop_name)
        crop.save(crop_path, quality=95)

        row_dict = row.to_dict()
        row_dict["path"] = crop_path
        row_dict["det_score"] = score
        row_dict["det_label"] = str(label)
        row_dict["is_crop"] = 1
        crop_rows.append(row_dict)

        if (idx + 1) % 200 == 0:
            print(f"processed {idx + 1}/{len(source_df)}")

    crop_df = pd.DataFrame(crop_rows)
    crop_csv_path = os.path.join(cfg.output_dir, "object_crop_train.csv")
    crop_df.to_csv(crop_csv_path, index=False)
    print(f"[crop csv] saved: {crop_csv_path}")
    display(crop_df.head(3))
else:
    print("Set BUILD_OBJECT_CROP=True to generate object crop dataset.")


## 7. 실험 팁

- **Smoke test**  
  `debug_max_samples=128`, `model_key="qwen2_5_vl_3b"`로 먼저 파이프라인이 끝까지 도는지 확인
- **정답 예측 방식**  
  최종 점수는 대개 `infer_mode="score"`가 더 안정적
- **체크포인트 재개**  
  `cfg.resume_from="latest"`로 바꾸면 가장 최근 checkpoint에서 이어서 학습
- **FlashAttention2**  
  A100/H100에서 설치가 되면 켜고, 안 되면 꺼도 됨
- **공개 리더보드용 최종본**  
  holdout 고정 → full-train 2회 → submission voting
